In [1]:
# --- CONFIGURACIÓN Y LIBRERÍAS ---
import pandas as pd
import numpy as np
import requests
import pyarrow as pa
import pyarrow.parquet as pq
import warnings
import os
import time
import subprocess
import socket
from sklearn.impute import SimpleImputer
from pathlib import Path

warnings.filterwarnings('ignore')

print("🚀 [PASO 0]: Preparando el entorno de trabajo...")
print("📦 Librerías cargadas (Pandas, Requests, PyArrow, Sklearn).")


🚀 [PASO 0]: Preparando el entorno de trabajo...
📦 Librerías cargadas (Pandas, Requests, PyArrow, Sklearn).


In [2]:
# --- CONTROL DE SERVIDOR  ---
def iniciar_servidor_automatico():
    
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    servidor_activo = sock.connect_ex(('127.0.0.1', 5000)) == 0
    sock.close()

    if not servidor_activo:
        print("📡 Servidor no detectado. Iniciando 'servidor_api.py' (Simulación API REST)...")
        subprocess.Popen(['python', 'servidor_api.py'], 
                         creationflags=subprocess.CREATE_NEW_CONSOLE if os.name == 'nt' else 0)
        
        url_check = "http://127.0.0.1:5000/health"
        vinculado = False
        while not vinculado:
            try:
                if requests.get(url_check, timeout=2).status_code == 200:
                    print("✅ API VINCULADA Y LISTA PARA LA INGESTA AUTOMATIZADA.")
                    vinculado = True
            except:
                time.sleep(2)
    else:
        print("✅ El servidor ya se encuentra activo y vinculado.")

iniciar_servidor_automatico()


📡 Servidor no detectado. Iniciando 'servidor_api.py' (Simulación API REST)...
✅ API VINCULADA Y LISTA PARA LA INGESTA AUTOMATIZADA.


In [3]:
# --- INGESTA DE DATOS (EXTRACT) ---
def pipeline_ingesta():
    
    url = "http://127.0.0.1:5000/api/v1/precios-glp"
    print("\n" + "="*50)
    print("📥 INGESTA DE DATOS (EXTRACT)")
    print("="*50)
    try:
        response = requests.get(url, timeout=None) 
        df = pd.DataFrame(response.json())
        df.columns = [c.lower().replace(" ", "_").strip() for c in df.columns]
        
        print(f"✅ Ingesta exitosa: {len(df)} registros recibidos desde la API REST.")
        return df
    except Exception as e:
        print(f"❌ Error en Ingesta: {e}"); return None

df_raw = pipeline_ingesta()


📥 INGESTA DE DATOS (EXTRACT)
✅ Ingesta exitosa: 786923 registros recibidos desde la API REST.


In [4]:
# --- SECCIÓN DE INYECCIÓN DE ERRORES ---

def inyectar_errores_pruebas(df):
    """
    Generación de datos erróneos intencionales (Nulos, Outliers, Negativos).
    Objetivo: Demostrar que las validaciones y reglas de negocio los detectan.
    """
    print("\n" + "="*50)
    print("🧪 INYECCIÓN DE ERRORES (SIMULACIÓN DE DATOS CORRUPTOS)")
    print("="*50)

    df_err = df.copy()
    col_p = [c for c in df_err.columns if 'precio' in c][0]

    # 1. Inyectar Nulos (Completitud)
    indices_nulos = df_err.sample(150).index
    df_err.loc[indices_nulos, col_p] = np.nan
  
    # 2. Inyectar Precios Negativos (Validez / Reglas de Negocio)
    indices_neg = df_err.sample(120).index
    df_err.loc[indices_neg, col_p] = -99.99

    # 3. Inyectar Outliers Extremos (Consistencia / IQR)
    indices_out = df_err.sample(110).index
    df_err.loc[indices_out, col_p] = 5555.55
  
    print(f"⚠️ Caos generado en '{col_p}': 150 nulos, 120 negativos y 110 outliers.")
    print("🔍 EVIDENCIA DE DATOS AFECTADOS (Muestra ampliada):")

    # Mostrar la corrupción de datos
    idx_all = indices_neg.union(indices_nulos).union(indices_out)
    display(df_err.loc[idx_all, ['departamento', 'provincia', 'distrito', col_p, 'marca']].head(10))
    return df_err

df_raw = inyectar_errores_pruebas(df_raw)


🧪 INYECCIÓN DE ERRORES (SIMULACIÓN DE DATOS CORRUPTOS)
⚠️ Caos generado en 'precio_de_venta_(soles)': 150 nulos, 120 negativos y 110 outliers.
🔍 EVIDENCIA DE DATOS AFECTADOS (Muestra ampliada):


,departamento,provincia,distrito,precio_de_venta_(soles),marca
969,LIMA,LIMA,PUEBLO LIBRE,NaN,
1547,LIMA,LIMA,BARRANCO,NaN,Solgas
1688,HUANUCO,LEONCIO PRADO,RUPA-RUPA,NaN,
3733,LIMA,LIMA,SAN MARTIN DE PORRES,NaN,Solgas
7776,LIMA,LIMA,BARRANCO,5555.55,Masgas
13187,SAN MARTIN,RIOJA,NUEVA CAJAMARCA,NaN,ENERGY GAS
13879,TACNA,TACNA,TACNA,NaN,
14900,PROV. CONST. DEL CALLAO,PROV. CONST. DEL CALLAO,CALLAO,NaN,
15711,AREQUIPA,AREQUIPA,MARIANO MELGAR,NaN,Primax Gas
16833,LA LIBERTAD,TRUJILLO,TRUJILLO,5555.55,Primax Gas


In [5]:
# --- MÉTRICAS DE CALIDAD ---
def auditoria_calidad_datos(df):
    """
    Requisito 2.1.2: Medición de Completitud, Unicidad, Validez, Consistencia y Actualidad.
    Se ejecuta ANTES de la limpieza para diagnosticar el estado del dataset.
    """
    print("\n" + "="*60)
    print("📊 REPORTE DE MÉTRICAS DE CALIDAD (REQUISITO 2.1.2)")
    print("="*60)
    
    df_audit = df.copy()
    col_p = [c for c in df_audit.columns if 'precio' in c][0]
    col_f = [c for c in df_audit.columns if 'fecha' in c][0]

    # 1. COMPLETITUD: % de valores no nulos por columna
    print(f"\n✅ MÉTRICA 1: COMPLETITUD")
    completitud = df_audit.notna().mean().round(4) * 100
    completitud_df = completitud.reset_index()
    completitud_df.columns = ['Columna', 'Completitud_%']
    display(completitud_df.sort_values('Completitud_%'))

    # 2. UNICIDAD: Detección de duplicados
    print(f"\n✅ MÉTRICA 2: UNICIDAD")
    n_dup = df_audit.duplicated().sum()
    n_total = len(df_audit)
    print(f"   Total registros: {n_total} | Duplicados: {n_dup}")
    print(f"   % Unicidad: {((n_total - n_dup) / n_total * 100):.2f}%")

    # 3. VALIDEZ: Rangos aceptables (Precio entre 5 y 300 soles)
    print(f"\n✅ MÉTRICA 3: VALIDEZ")
    precio_inv = df_audit[~df_audit[col_p].between(5, 300)]
    print(f"   Precios fuera de rango [5-300]: {len(precio_inv)} registros")

    # 4. CONSISTENCIA: Reglas lógicas (Precios no negativos)
    print(f"\n✅ MÉTRICA 4: CONSISTENCIA")
    negativos = df_audit[df_audit[col_p] <= 0]
    print(f"   Precios inconsistentes (<= 0): {len(negativos)} registros")

    # 5. ACTUALIDAD: Vigencia de los datos
    print(f"\n✅ MÉTRICA 5: ACTUALIDAD")
    df_audit[col_f] = pd.to_datetime(df_audit[col_f], errors='coerce')
    ultima_fecha = df_audit[col_f].max()
    print(f"   Último registro: {ultima_fecha} (Antigüedad: {(pd.Timestamp.now() - ultima_fecha).days} días)")

    # --- RESUMEN DE HALLAZGOS ---
    print("\n🎯 RESUMEN EJECUTIVO DE CALIDAD")
    resumen = {
        "Nulos detectados": df_audit.isna().sum().sum(),
        "Duplicados": n_dup,
        "Precios Inválidos": len(precio_inv) + len(negativos)
    }
    for k, v in resumen.items():
        icono = "🔴" if v > 0 else "✅"
        print(f"   {icono} {k}: {v}")
    
    return df_audit
df_audit = auditoria_calidad_datos(df_raw)


📊 REPORTE DE MÉTRICAS DE CALIDAD (REQUISITO 2.1.2)

✅ MÉTRICA 1: COMPLETITUD


,Columna,Completitud_%
6,precio_de_venta_(soles),99.98
0,actividad,100.00
2,dirección,100.00
1,departamento,100.00
3,distrito,100.00
4,fecha_de_registro,100.00
5,marca,100.00
7,producto,100.00
8,provincia,100.00
9,razón_social,100.00



✅ MÉTRICA 2: UNICIDAD
   Total registros: 786923 | Duplicados: 4394
   % Unicidad: 99.44%

✅ MÉTRICA 3: VALIDEZ
   Precios fuera de rango [5-300]: 7669 registros

✅ MÉTRICA 4: CONSISTENCIA
   Precios inconsistentes (<= 0): 120 registros

✅ MÉTRICA 5: ACTUALIDAD
   Último registro: 2026-04-12 23:51:44 (Antigüedad: 15 días)

🎯 RESUMEN EJECUTIVO DE CALIDAD
   🔴 Nulos detectados: 150
   🔴 Duplicados: 4394
   🔴 Precios Inválidos: 7789


In [6]:
# --- INTEGRACIÓN DE FUENTES ---
def integrar_fuentes_externas(df):
    
    # Aseguramos el formato de fecha para extraer los componentes
    df['fecha_de_registro'] = pd.to_datetime(df['fecha_de_registro'], errors='coerce')
    
    # Creamos las columnas temporales necesarias para el dashboard
    df['año'] = df['fecha_de_registro'].dt.year
    df['mes'] = df['fecha_de_registro'].dt.month
    
    print("\n" + "="*50)
    print("🧩 INTEGRACIÓN DE MÚLTIPLES FUENTES")
    print("="*50)
    distritos_unicos = df['distrito'].unique()
    
    # Fuente externa simulada 
    df_maestro = pd.DataFrame({
        'distrito_key': distritos_unicos,
        'id_region': [f"REG-{i}" for i in range(len(distritos_unicos))],
        'indice_demanda': np.random.uniform(0.5, 1.5, len(distritos_unicos))
    })
    
    df_integrado = pd.merge(df, df_maestro, left_on='distrito', right_on='distrito_key', how='left')
    
    print(f"✅ Integración completada. Columnas incorporadas: ['id_region', 'indice_demanda', 'año', 'mes']")
    print("🔍 TABLA DE ENRIQUECIMIENTO (Cruce con Maestro):")
    cols_evidencia = ['distrito', 'departamento', 'id_region', 'indice_demanda', 'año', 'mes']
    display(df_integrado[cols_evidencia].drop_duplicates().head(8))
    
    return df_integrado
df_integrado = integrar_fuentes_externas(df_raw)


🧩 INTEGRACIÓN DE MÚLTIPLES FUENTES
✅ Integración completada. Columnas incorporadas: ['id_region', 'indice_demanda', 'año', 'mes']
🔍 TABLA DE ENRIQUECIMIENTO (Cruce con Maestro):


,distrito,departamento,id_region,indice_demanda,año,mes
0,JAYANCA,LAMBAYEQUE,REG-0,1.429106,2025,1
3,RIMAC,LIMA,REG-1,0.839521,2025,1
4,ANCAHUASI,CUSCO,REG-2,1.204230,2025,1
5,SAN JUAN DE LURIGANCHO,LIMA,REG-3,0.996652,2025,1
8,ATE,LIMA,REG-4,1.375880,2025,1
10,JESUS MARIA,LIMA,REG-5,1.252663,2025,1
11,LURIN,LIMA,REG-6,1.154159,2025,1
12,LA VICTORIA,LIMA,REG-7,1.083531,2025,1


In [8]:
# --- CALIDAD E IMPUTACIÓN ---
def analisis_calidad_datos(df):
    
    print("\n" + "="*50)
    print("📊 CALIDAD DE DATOS E IMPUTACIÓN")
    print("="*50)
    
    df_c = df.copy()
    col_p = [c for c in df_c.columns if 'precio' in c][0]
    
    # --- ANTES: Identificación de registros corruptos ---
    idx_nulos = df_c[df_c[col_p].isna()].index
    nulos_count = len(idx_nulos)
    print(f"✔️ [ANTES]: Se detectaron {nulos_count} valores nulos (Falta de Completitud).")

    # --- PROCESO: Imputación Estadística ---
    mediana_calculada = df_c[col_p].median()
    imputer = SimpleImputer(strategy='median')
    df_c[[col_p]] = imputer.fit_transform(df_c[[col_p]])
    
    # --- DESPUÉS: Evidencia de la corrección ---
    print(f"✅ [DESPUÉS]: Nulos restaurados con la Mediana ({mediana_calculada:.2f}).")
    if nulos_count > 0:
        print("\n🔍 EVIDENCIA DE IMPUTACIÓN (Antes vs Después):")
        # Creamos una vista comparativa de los registros que eran nulos
        evidencia_nulos = df.loc[idx_nulos, [col_p]].rename(columns={col_p: 'Precio_Original (NaN)'})
        evidencia_nulos['Precio_Imputado'] = df_c.loc[idx_nulos, col_p]
        display(pd.concat([df_c.loc[idx_nulos, ['departamento', 'distrito', 'marca']], evidencia_nulos], axis=1).head(10))

    # --- LIMPIEZA DE MARCAS ---
    # Convertimos espacios en blanco a NaN y luego los vacíos a NaN para poder detectarlos
    df_c['marca'] = df_c['marca'].str.strip().replace('', np.nan)
    
    idx_nulos_m = df_c[df_c['marca'].isna()].index
    nulos_m_count = len(idx_nulos_m)
    print(f"\n✔️ [ANTES]: Se detectaron {nulos_m_count} marcas vacías o nulas.")
    
    df_c['marca'] = df_c['marca'].fillna('NO DECLARADO')
    print(f"✅ [DESPUÉS]: Marcas vacías/nulas reemplazadas por 'NO DECLARADO'.")
    
    if nulos_m_count > 0:
        display(df_c.loc[idx_nulos_m, ['departamento', 'distrito', 'marca']].head(10))

    # --- DETECCIÓN DE OUTLIERS ---
    Q1 = df_c[col_p].quantile(0.25); Q3 = df_c[col_p].quantile(0.75)
    IQR = Q3 - Q1
    limite_sup = Q3 + 1.5 * IQR
    
    outliers = df_c[df_c[col_p] > limite_sup]
    print(f"\n🚨 DETECCIÓN IQR: Se hallaron {len(outliers)} outliers (Valores > {limite_sup:.2f}).")
    if len(outliers) > 0:
        print("🔍 MUESTRA DE OUTLIERS DETECTADOS (Pendientes de validación de negocio):")
        display(outliers[['distrito', 'marca', col_p]].head(10))
    
    return df_c
df_calidad = analisis_calidad_datos(df_integrado)


📊 CALIDAD DE DATOS E IMPUTACIÓN
✔️ [ANTES]: Se detectaron 150 valores nulos (Falta de Completitud).
✅ [DESPUÉS]: Nulos restaurados con la Mediana (48.00).

🔍 EVIDENCIA DE IMPUTACIÓN (Antes vs Después):


,departamento,distrito,marca,Precio_Original (NaN),Precio_Imputado
969,LIMA,PUEBLO LIBRE,,NaN,48.0
1547,LIMA,BARRANCO,Solgas,NaN,48.0
1688,HUANUCO,RUPA-RUPA,,NaN,48.0
3733,LIMA,SAN MARTIN DE PORRES,Solgas,NaN,48.0
13187,SAN MARTIN,NUEVA CAJAMARCA,ENERGY GAS,NaN,48.0
13879,TACNA,TACNA,,NaN,48.0
14900,PROV. CONST. DEL CALLAO,CALLAO,,NaN,48.0
15711,AREQUIPA,MARIANO MELGAR,Primax Gas,NaN,48.0
19326,LIMA,SANTIAGO DE SURCO,,NaN,48.0
20364,CAJAMARCA,BELLAVISTA,Pecsagas,NaN,48.0



✔️ [ANTES]: Se detectaron 258190 marcas vacías o nulas.
✅ [DESPUÉS]: Marcas vacías/nulas reemplazadas por 'NO DECLARADO'.


,departamento,distrito,marca
5,LIMA,SAN JUAN DE LURIGANCHO,NO DECLARADO
6,LIMA,SAN JUAN DE LURIGANCHO,NO DECLARADO
7,LIMA,SAN JUAN DE LURIGANCHO,NO DECLARADO
10,LIMA,JESUS MARIA,NO DECLARADO
11,LIMA,LURIN,NO DECLARADO
12,LIMA,LA VICTORIA,NO DECLARADO
14,LIMA,PUENTE PIEDRA,NO DECLARADO
15,PROV. CONST. DEL CALLAO,CALLAO,NO DECLARADO
20,LIMA,SAN JUAN DE MIRAFLORES,NO DECLARADO
21,ANCASH,NUEVO CHIMBOTE,NO DECLARADO



🚨 DETECCIÓN IQR: Se hallaron 115051 outliers (Valores > 130.53).
🔍 MUESTRA DE OUTLIERS DETECTADOS (Pendientes de validación de negocio):


,distrito,marca,precio_de_venta_(soles)
2,JAYANCA,Solgas,220.00
19,SAN CLEMENTE,Llama Gas,208.00
29,HUAMACHUCO,Solgas,220.00
33,CHORRILLOS,Solgas,242.00
36,HUAMACHUCO,Solgas,215.00
49,LAMBAYEQUE,Primax Gas,260.89
50,LAMBAYEQUE,Pecsagas,260.89
51,LAMBAYEQUE,Progas,260.89
62,CERRO COLORADO,Primax Gas,250.87
63,CERRO COLORADO,Progas,250.87


In [9]:
# --- ENRIQUECIMIENTO Y NORMALIZACIÓN ---
def enriquecimiento_y_validación(df):
    
    print("\n" + "="*50)
    print("🛠️ ENRIQUECIMIENTO Y VALIDACIÓN (REGLAS DE NEGOCIO)")
    print("="*50)
    
    df_e = df.copy()
    col_p = [c for c in df_e.columns if 'precio' in c][0]
    total_inicial = len(df_e)

    # 1. Variables derivadas 
    col_f = [c for c in df_e.columns if 'fecha' in c][0]
    df_e[col_f] = pd.to_datetime(df_e[col_f], errors='coerce')
    df_e['trimestre'] = df_e[col_f].dt.quarter 

    # 2. VALIDACIÓN DE REGLAS DE NEGOCIO 
    limite_sup_negocio = 300
    alertas = df_e[(df_e[col_p] <= 0) | (df_e[col_p] > limite_sup_negocio)]
    
    print(f"🚩 [ANTES]: {len(alertas)} registros violan las reglas de negocio (Negativos o > {limite_sup_negocio}).")
    if len(alertas) > 0:
        print("🔍 EVIDENCIA DE DATOS ANÓMALOS DETECTADOS:")
        display(alertas[['distrito', 'marca', col_p, 'trimestre']].head(10))

    # --- PROCESO: Filtrado Efectivo ---
    df_e = df_e[(df_e[col_p] > 0) & (df_e[col_p] <= limite_sup_negocio)]
    
    # --- DESPUÉS: Resumen de limpieza ---
    print(f"\n✅ [DESPUÉS]: Limpieza completada.")
    print(f"📊 Resumen: {total_inicial} iniciales -> {len(df_e)} finales (Eliminados: {total_inicial - len(df_e)})")

    # 3. Binning y Normalización 
    df_e['rango_precio'] = pd.qcut(df_e[col_p], q=3, labels=['Bajo', 'Medio', 'Alto'], duplicates='drop')
    df_e['precio_norm'] = (df_e[col_p] - df_e[col_p].min()) / (df_e[col_p].max() - df_e[col_p].min())
    
    print("📈 EVIDENCIA DE NORMALIZACIÓN Y BINNING (Muestra del resultado final):")
    display(df_e[['distrito', col_p, 'rango_precio', 'precio_norm']].head(10))
    
    return df_e

df_enriquecido = enriquecimiento_y_validación(df_calidad)


🛠️ ENRIQUECIMIENTO Y VALIDACIÓN (REGLAS DE NEGOCIO)
🚩 [ANTES]: 1541 registros violan las reglas de negocio (Negativos o > 300).
🔍 EVIDENCIA DE DATOS ANÓMALOS DETECTADOS:


,distrito,marca,precio_de_venta_(soles),trimestre
79,LURIGANCHO,Primax Gas,317.70,1
1203,LURIGANCHO,Primax Gas,317.70,1
3106,LURIGANCHO,Primax Gas,317.70,1
4860,LURIGANCHO,Primax Gas,317.70,1
5758,LURIGANCHO,Primax Gas,317.70,1
7119,LURIGANCHO,Primax Gas,317.70,1
7776,BARRANCO,Masgas,5555.55,1
8827,LURIGANCHO,Primax Gas,317.70,1
10696,LURIGANCHO,Primax Gas,317.70,1
12528,LURIGANCHO,Primax Gas,317.70,1



✅ [DESPUÉS]: Limpieza completada.
📊 Resumen: 786923 iniciales -> 785382 finales (Eliminados: 1541)
📈 EVIDENCIA DE NORMALIZACIÓN Y BINNING (Muestra del resultado final):


,distrito,precio_de_venta_(soles),rango_precio,precio_norm
0,JAYANCA,50.00,Medio,0.162283
1,JAYANCA,53.00,Medio,0.172335
2,JAYANCA,220.00,Alto,0.731930
3,RIMAC,53.00,Medio,0.172335
4,ANCAHUASI,52.00,Medio,0.168984
5,SAN JUAN DE LURIGANCHO,8.79,Bajo,0.024193
6,SAN JUAN DE LURIGANCHO,7.69,Bajo,0.020507
7,SAN JUAN DE LURIGANCHO,7.69,Bajo,0.020507
8,ATE,57.50,Alto,0.187414
9,ATE,54.00,Alto,0.175686


In [10]:
# --- OPTIMIZACIÓN Y ALMACENAMIENTO ---
def optimizar_y_guardar(df):
    
    print("\n" + "="*50)
    print("🚀 OPTIMIZACIÓN Y EXPORTACIÓN")
    print("="*50)
    
    # Optimización de Memoria (Downcasting)
    mem_ini = df.memory_usage().sum()
    df[df.select_dtypes(include=['float64']).columns] = df.select_dtypes(include=['float64']).apply(pd.to_numeric, downcast='float')
    mem_fin = df.memory_usage().sum()
    print(f"📉 Downcasting: Ahorro de memoria RAM del {((mem_ini - mem_fin) / mem_ini * 100):.2f}%.")
    
    # Vectorización: Agregación rápida para validar integridad
    col_p = [c for c in df.columns if 'precio' in c][0]
    resumen = df.groupby('id_region')[col_p].agg(['mean', 'count']).head(5)
    print(f"📊 Resumen Vectorizado por Región:\n{resumen}")
    
   # Almacenamiento Optimizado 
    # '..' sube un nivel desde la carpeta 'notebooks' hacia la raíz 'PROYECTO_GLP'
    BASE_DIR = Path.cwd().parent

    output_dir = BASE_DIR / "data"
    output_dir.mkdir(parents=True, exist_ok=True)

    archivo = output_dir / "dataset_final_glp.parquet"
    
    df.to_parquet(archivo, engine='pyarrow', compression='snappy')
    print(f"💾 Éxito: Dataset almacenado en la ruta raíz: '{archivo}'.")
    return df

df_final = optimizar_y_guardar(df_enriquecido)

# Visualización final
if df_final is not None:
    print("\n" + "="*50)
    print("✅ DATASET FINAL SANEADO Y LISTO PARA ANÁLISIS")
    print("="*50)
    # Tabla amplia para revisión de resultados
    cols_finales = ['distrito', 'id_region', 'marca', 'precio_de_venta_(soles)', 'rango_precio', 'precio_norm', 'trimestre']
    display(df_final[cols_finales].head(15))


🚀 OPTIMIZACIÓN Y EXPORTACIÓN
📉 Downcasting: Ahorro de memoria RAM del 7.27%.
📊 Resumen Vectorizado por Región:
                 mean  count
id_region                   
REG-0       90.300880   1058
REG-1       72.555397   3519
REG-10      40.895466   7968
REG-100     60.071789   1811
REG-101    112.615273   1054
💾 Éxito: Dataset almacenado en la ruta raíz: 'c:\Users\tmigu\Desktop\Miguel\Ciclo 4\Módulo 7\5481 LENGUAJE DE CIENCIA DE DATOS II\Proyecto\proyecto_glp\data\dataset_final_glp.parquet'.

✅ DATASET FINAL SANEADO Y LISTO PARA ANÁLISIS


,distrito,id_region,marca,precio_de_venta_(soles),rango_precio,precio_norm,trimestre
0,JAYANCA,REG-0,Masgas,50.00,Medio,0.162283,1
1,JAYANCA,REG-0,Solgas,53.00,Medio,0.172335,1
2,JAYANCA,REG-0,Solgas,220.00,Alto,0.731930,1
3,RIMAC,REG-1,Primax Gas,53.00,Medio,0.172335,1
4,ANCAHUASI,REG-2,Somos Gas,52.00,Medio,0.168984,1
5,SAN JUAN DE LURIGANCHO,REG-3,NO DECLARADO,8.79,Bajo,0.024193,1
6,SAN JUAN DE LURIGANCHO,REG-3,NO DECLARADO,7.69,Bajo,0.020507,1
7,SAN JUAN DE LURIGANCHO,REG-3,NO DECLARADO,7.69,Bajo,0.020507,1
8,ATE,REG-4,Lima Gas,57.50,Alto,0.187414,1
9,ATE,REG-4,CASERITO,54.00,Alto,0.175686,1
